# 02 — Load Raw CSV Files into PostgreSQL

## Purpose and process

This notebook loads the validated synthetic CSV files into source-aligned tables in the PostgreSQL `raw` schema.

**Logic chain:** locate project → read local `.env` settings → verify target tables → truncate targets → copy CSV rows → confirm row counts.

Run `dw/sql/00_init_schemas.sql` and `dw/sql/01_raw_tables.sql` before this notebook.


## 1. Import libraries and locate the project

Only project-relative paths are used; no personal computer path is stored in the notebook.


In [ ]:
from datetime import date, timedelta
import os
from pathlib import Path

import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw" / "users.csv").exists():
            return candidate
    raise FileNotFoundError("Project root could not be located.")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT.name}")
print("Raw data directory: data/raw")


## 2. Read safe local database configuration

Credentials come from `.env`, which is excluded by `.gitignore`. The repository contains only `.env.example`.


In [ ]:
DB_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "customer_analytics_dw"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD"),
}

if not DB_CONFIG["password"]:
    raise RuntimeError("Set PGPASSWORD in the local .env file before loading data.")

print(f"Database target: {DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}")


## 3. Define repeatable CSV loading helpers

Each target is truncated before loading. This makes a full notebook rerun safe for the supplied static batch.


In [ ]:
def date_range(start_date: date, end_date: date):
    current = start_date
    while current <= end_date:
        yield current
        current += timedelta(days=1)


def ensure_table_exists(connection, schema_name: str, table_name: str) -> None:
    with connection.cursor() as cursor:
        cursor.execute("SELECT to_regclass(%s)", (f"{schema_name}.{table_name}",))
        if cursor.fetchone()[0] is None:
            raise RuntimeError(
                f"Missing target table {schema_name}.{table_name}. "
                "Run the schema SQL files first."
            )


def load_csv(
    connection,
    file_path: Path,
    schema_name: str,
    table_name: str,
    columns: tuple[str, ...],
) -> int:
    ensure_table_exists(connection, schema_name, table_name)

    table_ref = sql.SQL("{}.{}").format(
        sql.Identifier(schema_name),
        sql.Identifier(table_name),
    )
    column_list = sql.SQL(", ").join(map(sql.Identifier, columns))

    try:
        with connection.cursor() as cursor:
            cursor.execute(sql.SQL("TRUNCATE TABLE {}").format(table_ref))

            copy_statement = sql.SQL(
                "COPY {} ({}) FROM STDIN WITH (FORMAT CSV, HEADER TRUE)"
            ).format(table_ref, column_list)

            with file_path.open("r", encoding="utf-8-sig", newline="") as handle:
                cursor.copy_expert(copy_statement.as_string(connection), handle)

            cursor.execute(
                sql.SQL("UPDATE {} SET source_file = %s").format(table_ref),
                (file_path.name,),
            )
            cursor.execute(sql.SQL("SELECT COUNT(*) FROM {}").format(table_ref))
            row_count = cursor.fetchone()[0]

        connection.commit()
        return row_count
    except Exception:
        connection.rollback()
        raise


## 4. Load users, events, and orders

The connection is closed automatically after the batch completes.


In [ ]:
USER_COLUMNS = ("user_id", "registration_date", "region", "age_group")
EVENT_COLUMNS = (
    "event_id", "user_id", "event_type", "event_timestamp",
    "device", "channel", "session_id",
)
ORDER_COLUMNS = (
    "order_id", "user_id", "order_timestamp", "amount",
    "currency", "payment_method", "status",
)

load_results = []

with psycopg2.connect(**DB_CONFIG) as connection:
    count = load_csv(
        connection,
        RAW_DIR / "users.csv",
        "raw",
        "users_raw",
        USER_COLUMNS,
    )
    load_results.append(("users.csv", "raw.users_raw", count))

    for day in date_range(date(2025, 10, 1), date(2025, 10, 7)):
        event_file = RAW_DIR / f"daily_events_{day}.csv"
        event_table = f"events_{day:%Y%m%d}_raw"
        count = load_csv(connection, event_file, "raw", event_table, EVENT_COLUMNS)
        load_results.append((event_file.name, f"raw.{event_table}", count))

        order_file = RAW_DIR / f"daily_orders_{day}.csv"
        order_table = f"orders_{day:%Y%m%d}_raw"
        count = load_csv(connection, order_file, "raw", order_table, ORDER_COLUMNS)
        load_results.append((order_file.name, f"raw.{order_table}", count))

for source_file, target_table, row_count in load_results:
    print(f"{source_file:<36} -> {target_table:<30} {row_count:>5,} rows")

print("\nRAW LOAD COMPLETE")
